# 01 - Segment Power Calculations

For each mission segment, `evtolpy` computes three quantities following the power chain from the theory document (`docs/theory/evtol-sizing.md`):

**Step 1: Shaft Power** (kW): Mechanical power at the rotor shaft, computed from the segment specific force. The general form is:

$$P_{shaft,avg} = \frac{F_h \cdot v_h + F_v \cdot v_v}{\eta_{rotor}}$$

**Step 2: Electric Power** (kW): Electric power drawn from the battery, accounting for EPU losses:

$$P_{elec,avg} = \frac{P_{shaft,avg}}{\eta_{epu}}$$

**Step 3: Energy** (kWh): Total energy consumed in the segment:

$$E = P_{elec,avg} \cdot \frac{t}{3600}$$

**Step 3: Energy** ADD C-rate

In [ ]:
import sys
sys.path.append('../../evtol')
from aircraft import Aircraft

aircraft = Aircraft('../Archer_Midnight.json')

: 

## Hover Power

Hover power is derived from **momentum theory**. The thrust required to sustain the aircraft's weight in a stationary hover is $T = m \cdot g$. The induced velocity through the rotor disk is:

$$v_{i} = \sqrt{\frac{T}{2 \cdot \rho \cdot A_{disk}}}$$

The ideal hover power is then $P_{hover} = T \cdot v_i$, and the shaft power accounts for rotor efficiency:

$$P_{shaft} = \frac{T \cdot v_i}{\eta_{rotor}}$$

This shows why **disk loading** ($m/A_{disk}$) is important, lower disk loading (larger rotors) reduces induced velocity and thus hover power. It also shows why hover power scales with $m^{3/2}$, doubling the aircraft mass more than doubles the hover power. As every problem in engineering it falls down to an optimzation problem, the larger the rotor the heavier but the more efficient.

In [ ]:
print(f"Hover Shaft Power:    {aircraft.hover_shaft_power_kw:.2f} kW")
print(f"Hover Electric Power: {aircraft.hover_shaft_power_kw / aircraft.power.epu_effic:.2f} kW")
print(f"")
print(f"  MTOW:         {aircraft.max_takeoff_mass_kg:.0f} kg")
print(f"  Disk Area:    {aircraft.propulsion.disk_area_m2:.2f} m^2")
print(f"  Air Density:  {aircraft.environ.air_density_sea_lvl_kg_p_m3:.3f} kg/m^3")

## Primary Mission - Per-Segment Power and Energy

In [ ]:
primary = [
    ('Depart Taxi',         aircraft.depart_taxi_avg_shaft_power_kw,     aircraft.depart_taxi_avg_electric_power_kw,     aircraft.depart_taxi_energy_kw_hr),
    ('Hover Climb',         aircraft.hover_climb_avg_shaft_power_kw,     aircraft.hover_climb_avg_electric_power_kw,     aircraft.hover_climb_energy_kw_hr),
    ('Transition Climb',    aircraft.trans_climb_avg_shaft_power_kw,     aircraft.trans_climb_avg_electric_power_kw,     aircraft.trans_climb_energy_kw_hr),
    ('Depart Procedures',   aircraft.depart_proc_avg_shaft_power_kw,    aircraft.depart_proc_avg_electric_power_kw,    aircraft.depart_proc_energy_kw_hr),
    ('Accelerate Climb',    aircraft.accel_climb_avg_shaft_power_kw,     aircraft.accel_climb_avg_electric_power_kw,     aircraft.accel_climb_energy_kw_hr),
    ('Cruise',              aircraft.cruise_avg_shaft_power_kw,          aircraft.cruise_avg_electric_power_kw,          aircraft.cruise_energy_kw_hr),
    ('Decelerate Descend',  aircraft.decel_descend_avg_shaft_power_kw,   aircraft.decel_descend_avg_electric_power_kw,   aircraft.decel_descend_energy_kw_hr),
    ('Arrive Procedures',   aircraft.arrive_proc_avg_shaft_power_kw,    aircraft.arrive_proc_avg_electric_power_kw,    aircraft.arrive_proc_energy_kw_hr),
    ('Transition Descend',  aircraft.trans_descend_avg_shaft_power_kw,   aircraft.trans_descend_avg_electric_power_kw,   aircraft.trans_descend_energy_kw_hr),
    ('Hover Descend',       aircraft.hover_descend_avg_shaft_power_kw,   aircraft.hover_descend_avg_electric_power_kw,   aircraft.hover_descend_energy_kw_hr),
    ('Arrive Taxi',         aircraft.arrive_taxi_avg_shaft_power_kw,     aircraft.arrive_taxi_avg_electric_power_kw,     aircraft.arrive_taxi_energy_kw_hr),
]

print(f"{'Segment':<25s} {'Shaft (kW)':>10s} {'Elec (kW)':>10s} {'Energy (kWh)':>12s}")
print("-" * 60)
total_energy = 0.0
for name, sp, ep, en in primary:
    print(f"{name:<25s} {sp:>10.2f} {ep:>10.2f} {en:>12.4f}")
    total_energy += en
print("-" * 60)
print(f"{'Total Primary':<25s} {'':>10s} {'':>10s} {total_energy:>12.4f}")

In [ ]:
import matplotlib.pyplot as plt

seg_names = [name for name, _, _, _ in primary]
shaft_powers = [sp for _, sp, _, _ in primary]
elec_powers = [ep for _, _, ep, _ in primary]
energies = [en for _, _, _, en in primary]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Shaft vs Electric power
x = range(len(seg_names))
w = 0.35
axes[0].bar([i - w/2 for i in x], shaft_powers, w, label='Shaft', color='steelblue')
axes[0].bar([i + w/2 for i in x], elec_powers, w, label='Electric', color='coral')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(seg_names, rotation=60, ha='right', fontsize=7)
axes[0].set_ylabel('Power (kW)')
axes[0].set_title('Shaft vs Electric Power')
axes[0].legend(fontsize=8)

# Energy by segment
colors = ['#e15759' if e == max(energies) else '#4e79a7' for e in energies]
axes[1].bar(seg_names, energies, color=colors, edgecolor='white')
axes[1].set_xticklabels(seg_names, rotation=60, ha='right', fontsize=7)
axes[1].set_ylabel('Energy (kWh)')
axes[1].set_title('Energy by Segment')

# EPU efficiency loss
epu_loss = [ep - sp for sp, ep in zip(shaft_powers, elec_powers)]
axes[2].bar(seg_names, shaft_powers, label='Useful shaft power', color='steelblue')
axes[2].bar(seg_names, epu_loss, bottom=shaft_powers, label=f'EPU loss ({1-aircraft.power.epu_effic:.0%})', color='#f28e2b', alpha=0.7)
axes[2].set_xticklabels(seg_names, rotation=60, ha='right', fontsize=7)
axes[2].set_ylabel('Power (kW)')
axes[2].set_title('Power Chain Losses')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

## Reserve Mission - Per-Segment Energy

The reserve mission must be sized for the same aircraft but a short diversion route. The battery must carry enough energy for both the primary and reserve missions. It should be noted that the required regulatory diversion route has not yet been specified for eVTOLs yet. For a conventional aircraft this would involve a 45 minute cruise reserve, a requirement impossible to meet for this aircraft type. 

In [ ]:
reserve = [
    ('Rsv Hover Climb',     aircraft.reserve_hover_climb_energy_kw_hr),
    ('Rsv Trans. Climb',    aircraft.reserve_trans_climb_energy_kw_hr),
    ('Rsv Accel. Climb',    aircraft.reserve_accel_climb_energy_kw_hr),
    ('Rsv Cruise',          aircraft.reserve_cruise_energy_kw_hr),
    ('Rsv Decel. Descend',  aircraft.reserve_decel_descend_energy_kw_hr),
    ('Rsv Trans. Descend',  aircraft.reserve_trans_descend_energy_kw_hr),
    ('Rsv Hover Descend',   aircraft.reserve_hover_descend_energy_kw_hr),
]

reserve_total = sum(en for _, en in reserve)
print(f"{'Segment':<25s} {'Energy (kWh)':>12s}")
print("-" * 38)
for name, en in reserve:
    print(f"{name:<25s} {en:>12.4f}")
print("-" * 38)
print(f"{'Total Reserve':<25s} {reserve_total:>12.4f}")
print(f"")
print(f"Total Mission Energy (primary + reserve): {total_energy + reserve_total:.4f} kWh")

## Summary

The three step power chain converts physical forces into energy consumption:

$$\text{Forces} \xrightarrow{\div \eta_{rotor}} P_{shaft} \xrightarrow{\div \eta_{epu}} P_{elec} \xrightarrow{\times t} E$$

Key observations from the Archer Midnight:

- **Hover segments** have the highest instantaneous power due to momentum theory ($P \propto m^{3/2}$)
- **Cruise** has moderate power but dominates total energy because of its long duration
- The **EPU efficiency loss** (assumed constant percentage) is proportionally larger for high power segments
- **Reserve energy** adds a significant fraction to the total battery requirement which directly increases MTOW

**Next:** [02 - Power and Energy Visualization]